# 02 - Vision Transformer (ViT)

**Goal:** Understand how transformers are applied to images.

---

## The Challenge

Transformers were designed for sequences (text). Images are 2D grids.

**Problem:** A 224x224 image = 50,176 pixels. Attention between all pairs = 2.5 billion operations!

**Solution:** Divide image into **patches**, treat patches as tokens.

## ViT Architecture

```
Image (224x224)
    ↓
Split into patches (14x14 = 196 patches of 16x16)
    ↓
Flatten each patch (16x16x3 = 768 values)
    ↓
Linear projection → patch embeddings
    ↓
Add positional embeddings
    ↓
Add [CLS] token
    ↓
Transformer encoder (self-attention layers)
    ↓
[CLS] token → classification head
```

Now attention is between 196 patches instead of 50,176 pixels!

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

## Patch Embedding

Split image into non-overlapping patches and project them:

In [ ]:
class PatchEmbedding(nn.Module):
    """Split image into patches and embed them."""
    
    def __init__(self, img_size=224, patch_size=16, in_channels=3, embed_dim=768):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2
        
        # Use conv with kernel=stride=patch_size to extract patches
        self.projection = nn.Conv2d(
            in_channels, embed_dim,
            kernel_size=patch_size, stride=patch_size
        )
    
    def forward(self, x):
        # x: (batch, channels, height, width)
        x = self.projection(x)  # (batch, embed_dim, h/patch, w/patch)
        x = x.flatten(2)         # (batch, embed_dim, num_patches)
        x = x.transpose(1, 2)    # (batch, num_patches, embed_dim)
        return x

# Test
patch_embed = PatchEmbedding(img_size=224, patch_size=16, embed_dim=768)
x = torch.randn(1, 3, 224, 224)  # RGB image

patches = patch_embed(x)
print(f"Input image: {x.shape}")
print(f"Patch embeddings: {patches.shape}")
print(f"Number of patches: {patch_embed.num_patches} (14x14)")

In [ ]:
# Visualize patches
def visualize_patches(image, patch_size=16):
    """Show how an image is divided into patches."""
    h, w = image.shape[:2]
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Original image with grid
    axes[0].imshow(image, cmap='gray')
    for i in range(0, h + 1, patch_size):
        axes[0].axhline(y=i, color='red', linewidth=0.5)
    for j in range(0, w + 1, patch_size):
        axes[0].axvline(x=j, color='red', linewidth=0.5)
    axes[0].set_title(f'Image divided into {patch_size}x{patch_size} patches')
    axes[0].axis('off')
    
    # Show some patches
    num_h = h // patch_size
    num_w = w // patch_size
    
    # Create grid of patches
    patch_grid = np.zeros((num_h * (patch_size + 2), num_w * (patch_size + 2)))
    for i in range(num_h):
        for j in range(num_w):
            patch = image[i*patch_size:(i+1)*patch_size, j*patch_size:(j+1)*patch_size]
            pi = i * (patch_size + 2) + 1
            pj = j * (patch_size + 2) + 1
            patch_grid[pi:pi+patch_size, pj:pj+patch_size] = patch
    
    axes[1].imshow(patch_grid, cmap='gray')
    axes[1].set_title('Patches (separated)')
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()

# Create a simple test image
test_img = np.random.rand(64, 64)
# Add some structure
test_img[20:44, 20:44] = 0.8
test_img[28:36, 28:36] = 0.2

visualize_patches(test_img, patch_size=16)

## Transformer Encoder Block

Each block has:
1. Layer Norm + Multi-Head Self-Attention
2. Layer Norm + Feed-Forward Network (MLP)
3. Residual connections

In [ ]:
class TransformerBlock(nn.Module):
    """One transformer encoder block."""
    
    def __init__(self, embed_dim, num_heads, mlp_ratio=4.0, dropout=0.0):
        super().__init__()
        
        # Attention
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        
        # MLP
        self.norm2 = nn.LayerNorm(embed_dim)
        mlp_hidden = int(embed_dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden, embed_dim),
            nn.Dropout(dropout),
        )
    
    def forward(self, x):
        # Attention with residual
        x_norm = self.norm1(x)
        attn_out, _ = self.attn(x_norm, x_norm, x_norm)
        x = x + attn_out
        
        # MLP with residual
        x = x + self.mlp(self.norm2(x))
        
        return x

# Test
block = TransformerBlock(embed_dim=768, num_heads=12)
x = torch.randn(2, 196, 768)  # batch=2, 196 patches, dim=768

output = block(x)
print(f"Input: {x.shape}")
print(f"Output: {output.shape}")
print(f"Parameters: {sum(p.numel() for p in block.parameters()):,}")

## Complete ViT for Classification

In [ ]:
class ViT(nn.Module):
    """Vision Transformer for image classification."""
    
    def __init__(
        self,
        img_size=224,
        patch_size=16,
        in_channels=3,
        num_classes=1000,
        embed_dim=768,
        depth=12,
        num_heads=12,
    ):
        super().__init__()
        
        # Patch embedding
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, embed_dim)
        num_patches = self.patch_embed.num_patches
        
        # CLS token (learnable)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        
        # Positional embedding (learnable)
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))
        
        # Transformer encoder
        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads)
            for _ in range(depth)
        ])
        
        self.norm = nn.LayerNorm(embed_dim)
        
        # Classification head
        self.head = nn.Linear(embed_dim, num_classes)
        
        # Initialize
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
    
    def forward(self, x):
        batch = x.shape[0]
        
        # Patch embedding
        x = self.patch_embed(x)  # (batch, num_patches, embed_dim)
        
        # Add CLS token
        cls_tokens = self.cls_token.expand(batch, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)  # (batch, num_patches + 1, embed_dim)
        
        # Add positional embedding
        x = x + self.pos_embed
        
        # Transformer blocks
        for block in self.blocks:
            x = block(x)
        
        x = self.norm(x)
        
        # Use CLS token for classification
        cls_output = x[:, 0]
        
        return self.head(cls_output)

# Create a small ViT
vit = ViT(
    img_size=224,
    patch_size=16,
    num_classes=10,
    embed_dim=384,  # Smaller for demo
    depth=6,
    num_heads=6,
)

x = torch.randn(2, 3, 224, 224)
output = vit(x)

print(f"Input: {x.shape}")
print(f"Output: {output.shape}")
print(f"Total parameters: {sum(p.numel() for p in vit.parameters()):,}")

## ViT Sizes

| Model | Layers | Hidden | Heads | Params |
|-------|--------|--------|-------|--------|
| ViT-Tiny | 12 | 192 | 3 | 5.7M |
| ViT-Small | 12 | 384 | 6 | 22M |
| ViT-Base | 12 | 768 | 12 | 86M |
| ViT-Large | 24 | 1024 | 16 | 307M |
| ViT-Huge | 32 | 1280 | 16 | 632M |

## ViT Limitations

1. **Fixed resolution** - trained on specific size, doesn't generalize well
2. **No hierarchy** - all patches at same scale (unlike CNN: edges → shapes → objects)
3. **Needs lots of data** - transformers need more data than CNNs to train well

**Swin Transformer** fixes these issues!

## Summary

| Component | Purpose |
|-----------|--------|
| **Patches** | Split image into tokens (e.g., 16x16) |
| **Patch embedding** | Linear projection of flattened patches |
| **CLS token** | Special token for classification |
| **Positional embedding** | Add position information |
| **Transformer blocks** | Self-attention + MLP, stacked |

**Next:** Swin Transformer - efficient attention with shifted windows.